# PO-000 Reviewer Walkthrough

**Paper:** *Projection Obstruction for Factorized States* (PO-000) — `projection_obstruction.md` at the package root.
**AuthorityEffect:** None — this notebook checks receipts; it authorizes nothing.

This is a deliberately thin walkthrough for a reviewer: it runs the deterministic
mechanization replay, verifies the resulting certificate with the stdlib-only
verifier, and resolves the sample claim-to-code map so you can see where each
claim's evidence lives. It is **not** the falsification sandbox — for interactive
attack surfaces open `falsification_sandbox.ipynb`.

*Provenance note (2026-08-31): an earlier file at this name was a byte-identical
duplicate of `falsification_sandbox.ipynb`; it was replaced by this genuine
walkthrough. See `projection_obstruction_change_log.md`.*

In [ ]:
# 1. Run the deterministic mechanization replay (fast: ~1 s)
import os
import subprocess
import sys
from pathlib import Path

PKG = (Path.cwd() if (Path.cwd() / "code").exists() else Path.cwd().parent).resolve()
sys.path.insert(0, str(PKG / "code"))
sys.path.insert(0, str(PKG / "code" / "tests"))

env = dict(os.environ, PYTHONPATH=str(PKG / "code"))
# Write outside the package tree so SHA256SUMS stays clean after this run.
import tempfile
cert_path = Path(tempfile.mkdtemp(prefix="po000_reviewer_")) / "000_mechanization_run.json"
result = subprocess.run(
    [sys.executable, str(PKG / "code" / "replay.py"), "--output", str(cert_path)],
    env=env, capture_output=True, text=True, cwd=str(PKG),
)
print("replay exit:", result.returncode)
assert result.returncode == 0, result.stderr[-2000:]
print("certificate written:", cert_path.name)

In [ ]:
# 2. Independently verify the certificate (stdlib-only verifier).
# This checks result totality against the claim ledger, source and
# publication hash inventories (required files must exist), and verdicts.
import json

verify = subprocess.run(
    [sys.executable, str(PKG / "code" / "verify_certificate.py"), str(cert_path)],
    capture_output=True, text=True, cwd=str(PKG),
)
report = json.loads(verify.stdout)
print(json.dumps(report, indent=2))
assert verify.returncode == 0 and report["verdict"] == "pass", report

In [ ]:
# 3. Resolve claims to code against the CANONICAL ledger.
#
# 2026-09-03: this cell used to parse docs/claim_code_map.yaml, a hand-kept
# sample map. An external review found it assigned the wrong meaning to every
# one of D1-D4 -- it called D1 the short-time defect theorem, while the ledger
# defines D1 as the RE1-RE5 structural declarations -- and overstated statuses
# as MECHANIZED. Rather than repair a derivative map that can drift again, the
# map was deleted and this cell reads the canonical ledger directly.
#
# Plain line-parse, deliberately: the deposit runs on stdlib + numpy + pytest,
# and this notebook does not add a YAML dependency to check a YAML file.
import importlib

text = (PKG / "projection_obstruction_claim_ledger.yaml").read_text(encoding="utf-8")

claim_id, pending, rows = None, {}, []
for raw in text.splitlines():
    line = raw.strip()
    if line.startswith("- claim_id:"):
        claim_id = line.split(":", 1)[1].strip().strip('"')
    elif line.startswith('- kind: "python"'):
        pending = {"claim_id": claim_id}
        rows.append(pending)
    elif pending is not None and line.startswith("- kind:"):
        pending = {}
    elif pending and ":" in line:
        key, _, value = line.lstrip("- ").partition(":")
        if key.strip() in {"module_path", "file_location"}:
            pending[key.strip()] = value.strip().strip('"')

print("python evidence pointers in the canonical ledger:", len(rows))

broken = 0
for row in rows:
    location = row.get("file_location")
    module = (row.get("module_path") or "").replace("code.", "").replace("tests.", "")
    exists = bool(location) and (PKG / location).is_file()
    try:
        importlib.import_module(module) if module else None
        imports = True
    except Exception:
        imports = False
    if not (exists and imports):
        broken += 1
        print(f"  BROKEN  {row['claim_id']:14} {location} "
              f"(exists={exists}, imports={imports})")

print(f"{len(rows)} checked, {broken} broken")
assert broken == 0, "the canonical ledger names Python evidence that does not resolve"


## 4. Where the rest of the evidence lives

The map above is a four-row **sample**. The complete evidence surfaces are:

| Surface | Path | How to check |
|---|---|---|
| Full typed claim ledger (~40 rows) | `projection_obstruction_claim_ledger.md` | read; every row names its evidence |
| Executable claim-coverage matrix | `code/claim_coverage.py` | `code/replay.py` checks totality on every run (step 1 above) |
| Lean 4 mechanization map | sources under `lean4/`, with per-file status in `lean4/README.md` and `lean4/manifest.json` | see `lean4/README.md` for status labels (including the LEAN_TOY residuals) |
| Full test suite | `code/tests/` | `python -m pytest code/tests` from the package root |
| Byte-integrity manifest | `SHA256SUMS` | `python scripts/regen_sha256sums.py --check` |
| Version history, incl. repairs | `projection_obstruction_change_log.md` | read newest entries first |

AuthorityEffect: None — a green cell here certifies bytes and receipts, never a mathematical claim beyond what the named evidence itself establishes.